In [50]:
from deck import full_euchre_deck
from dealer import Dealer
# from numba import njit
from tree_search import definitive_winner
import numpy as np


In [51]:
evaluate = np.array(
    [  # left bower  (Jack of clubs)
        [0, 140],
        [0, 135],    
        [0, -9],
        [-9, 0], 
        [9, 0]
    ]
)
up_card = np.array(
    [  # left bower  (Jack of clubs)
        [0, 90],  
    ]
)

In [52]:
def generate_hands(
    n_games=100,
    stack: np.array = None,
    stack_player: int = None,
    up_card: np.array = None,
    up_card_player: int = None,
):
    """Generate n random hand configurations."""
    all_hands = np.zeros((n_games, 4, 5, 2), dtype=np.int64)

    for i in range(n_games):
        game = Dealer(deck=full_euchre_deck, players=4)
        if stack is not None:
            game.stack_deck(stack_cards=stack, player=stack_player)
        if up_card is not None:
            game.stack_deck(stack_cards=up_card, player=up_card_player)
        game.deal_cards()
        all_hands[i] = np.array([game.hand1, game.hand2, game.hand3, game.hand4])

    return all_hands

In [53]:
game = Dealer(deck=full_euchre_deck ,players=4)
game.stack_deck(stack_cards=evaluate, player=1)
game.stack_deck(stack_cards=up_card, player=2)
game.deal_cards()
hands_test = np.array([game.hand1, game.hand2, game.hand3, game.hand4])
hands_test

array([[[  0, 140],
        [  0, 135],
        [  0,  -9],
        [ -9,   0],
        [  9,   0]],

       [[  0,  90],
        [  0, -10],
        [ 13,   0],
        [  0, -14],
        [ 10,   0]],

       [[ 12,   0],
        [-12,   0],
        [  0, 110],
        [  0, 120],
        [  0, -13]],

       [[  0, -12],
        [-14,   0],
        [-13,   0],
        [ 11,   0],
        [  0, 100]]])

In [54]:
# quick sanity check on the definitive winner function with a known hand configuration
definitive_winner(dealt_hands=hands_test, starting_player=3, caller=0, verbose=True)

[ 0.28055556 -0.13157895 -0.13157895  0.37531172  0.06896552]
[ 1.         -0.45714286 -0.45714286  1.        ]
[ 1.  -2.  -0.7]
[-2. -2.]
[-2.]
Starting hands:
 [[[  0 140]
  [  0 135]
  [  0  -9]
  [ -9   0]
  [  9   0]]

 [[  0  90]
  [  0 -10]
  [ 13   0]
  [  0 -14]
  [ 10   0]]

 [[ 12   0]
  [-12   0]
  [  0 110]
  [  0 120]
  [  0 -13]]

 [[  0 -12]
  [-14   0]
  [-13   0]
  [ 11   0]
  [  0 100]]]
Trick 1:
 [[ -9   0]
 [  0  90]
 [-12   0]
 [-14   0]]
Trick 1 winner: 1
next round hands:
 [[[  0 140]
  [  0 135]
  [  0  -9]
  [  9   0]]

 [[  0 -10]
  [ 13   0]
  [  0 -14]
  [ 10   0]]

 [[ 12   0]
  [  0 110]
  [  0 120]
  [  0 -13]]

 [[  0 -12]
  [-13   0]
  [ 11   0]
  [  0 100]]]
Trick 2: [[ 9  0]
 [13  0]
 [12  0]
 [11  0]]
Trick 2 winner: 1
next round hands:
 [[[  0 140]
  [  0 135]
  [  0  -9]]

 [[  0 -10]
  [  0 -14]
  [ 10   0]]

 [[  0 110]
  [  0 120]
  [  0 -13]]

 [[  0 -12]
  [-13   0]
  [  0 100]]]
Trick 3:
 [[  0  -9]
 [  0 -14]
 [  0 -13]
 [  0 -12]]
Trick 3 

-2

In [55]:
gen_test = generate_hands(n_games=500, stack=evaluate, stack_player=1, up_card=up_card, up_card_player=2)

In [56]:
scores = np.zeros(500, dtype=np.int64)

for i in range(500):
    scores[i] = definitive_winner(dealt_hands=gen_test[i], starting_player=0, caller=0, verbose=False)

mean_score = np.mean(scores)
se = np.std(scores, ddof=1) / np.sqrt(len(scores))

ci_lower = mean_score - 1.96 * se
ci_upper = mean_score + 1.96 * se

print(f"Expected value: {mean_score:.3f}")
print(f"95% CI: [{ci_lower:.3f}, {ci_upper:.3f}]")

Expected value: 0.188
95% CI: [0.059, 0.317]
